# Importing the dependencies/Libraries

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [34]:
!pip install scikit-learn-extra

In [35]:
!pip install "numpy<2"

In [36]:
from sklearn_extra.cluster import KMedoids

# Importing yahoo finance with alias

In [37]:
# importing yahoo finance
!pip install yfinance
import yfinance as yf

# Defining and downloading the ticks for the top 30 components in nasdaq 100 index

In [38]:
# A list of the top 30 components

ticks = ['AAPL', 'ALNY', 'AMAT', 'AMGN', 'APP', 'ARM', 'BKR', 'CCEP', 'CEG',
         'CSGP', 'DDOG', 'FTNT', 'HON', 'INSM', 'INTU', 'KDP', 'MAR', 'MCHP',
         'MDLZ', 'MELI', 'META', 'MRVL', 'PCAR', 'SBUX', 'STX', 'TSLA',
         'VRSK', 'WDC', 'WMT', 'ZS']


df_weekly = yf.download(ticks, period = '1y', interval='1d')

df_weekly.head()

/tmp/ipykernel_1156/1452635401.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_weekly = yf.download(ticks, period = '1y', interval='1d')
[*********************100%***********************]  30 of 30 completed


Price            Close                                                  \
Ticker            AAPL        ALNY        AMAT        AMGN         APP   
Date                                                                     
2025-04-03  202.308258  262.160004  134.308502  300.424927  261.980011   
2025-04-04  187.562531  235.740005  125.824387  285.435181  219.369995   
2025-04-07  180.672562  232.949997  131.672073  280.878113  232.220001   
2025-04-08  171.671783  224.320007  127.816574  271.734985  235.279999   
2025-04-09  197.987091  243.270004  148.402420  282.235565  274.959991   

Price                                                                ...  \
Ticker             ARM        BKR       CCEP         CEG       CSGP  ...   
Date                                                                 ...   
2025-04-03   97.720001  40.048744  87.512321  189.280563  76.349998  ...   
2025-04-04   87.709999  34.706951  81.773643  170.097809  72.620003  ...   
2025-04-07   88.629997  35.000992  80.857796  178.883240  75.750000  ...   
2025-04-08   85.820000  34.305088  79.737343  184.007294  73.230003  ...   
2025-04-09  106.589996  37.980637  82.903839  214.363419  78.699997  ...   

Price         Volume                                                    \
Ticker          META      MRVL     PCAR      SBUX       STX       TSLA   
Date                                                                     
2025-04-03  34777500  25213200  4165800  20490100  12915600  136174300   
2025-04-04  38589800  37313400  6326900  19700100   9249900  181229400   
2025-04-07  36606100  30875500  5586000  26099400   7297500  183453800   
2025-04-08  28034200  32326200  5344600  20534600   5679900  171603500   
2025-04-09  39216600  38033700  6972300  26668100  10253500  219433400   

Price                                             
Ticker         VRSK       WDC       WMT       ZS  
Date                                              
2025-04-03  1425300  21307200  33012900  3898100  
2025-04-04  2102300  22939100  36209000  5710500  
2025-04-07  1815500  14472100  36884900  5738300  
2025-04-08  1852800  17547200  34351700  2975300  
2025-04-09  1965700  20134800  46632800  4509500  

[5 rows x 150 columns]

In [39]:
df_weekly.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 251 entries, 2025-04-03 to 2026-04-02
Columns: 150 entries, ('Close', 'AAPL') to ('Volume', 'ZS')
dtypes: float64(120), int64(30)
memory usage: 296.1 KB


# Transposing the dataframe for clustering and reduction

In [40]:
print(df_weekly.columns)

MultiIndex([( 'Close', 'AAPL'),
            ( 'Close', 'ALNY'),
            ( 'Close', 'AMAT'),
            ( 'Close', 'AMGN'),
            ( 'Close',  'APP'),
            ( 'Close',  'ARM'),
            ( 'Close',  'BKR'),
            ( 'Close', 'CCEP'),
            ( 'Close',  'CEG'),
            ( 'Close', 'CSGP'),
            ...
            ('Volume', 'META'),
            ('Volume', 'MRVL'),
            ('Volume', 'PCAR'),
            ('Volume', 'SBUX'),
            ('Volume',  'STX'),
            ('Volume', 'TSLA'),
            ('Volume', 'VRSK'),
            ('Volume',  'WDC'),
            ('Volume',  'WMT'),
            ('Volume',   'ZS')],
           names=['Price', 'Ticker'], length=150)


In [41]:
# manipulating the dataframe shape for clustering
df_stacked = df_weekly.stack(level=1, future_stack=True)

df_final = df_stacked.reset_index(level=1)
df_final.head()

Price,Ticker,Close,High,Low,Open,Volume
Date,,,,,,
2025-04-03,AAPL,202.308258,206.589601,200.376674,204.648051,103419000
2025-04-03,ALNY,262.160004,264.649994,258.649994,263.000000,1028400
2025-04-03,AMAT,134.308502,140.612112,134.199477,139.749831,11779600
2025-04-03,AMGN,300.424927,305.321303,297.370750,301.714458,3280300
2025-04-03,APP,261.980011,268.000000,249.080002,264.519989,8959500


In [42]:
df_final.info()

df_final_c = df_final.groupby('Ticker', as_index=False).mean()


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 7530 entries, 2025-04-03 to 2026-04-02
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Ticker  7530 non-null   object 
 1   Close   7530 non-null   float64
 2   High    7530 non-null   float64
 3   Low     7530 non-null   float64
 4   Open    7530 non-null   float64
 5   Volume  7530 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 411.8+ KB


In [43]:
df_final_c.index

RangeIndex(start=0, stop=30, step=1)

# Using the close prices for clustering to choose tickers

In [44]:
# subsetting the columns to fit the pipeline

categorical_cols = []

numerical_cols = []

for col in df_final_c.columns:
  if df_final[col].dtype == 'object':
    categorical_cols.append(col)
  else:
    numerical_cols.append(col)

print(categorical_cols)
print(numerical_cols)

['Ticker']
['Close', 'High', 'Low', 'Open', 'Volume']


In [45]:
# Creating the pipeline for the data

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols)
])

pipeline = Pipeline([('data preprocessing', preprocessor),
    ('reduction', PCA(n_components=2)),
    ('kmedoids', KMedoids (n_clusters=4, random_state=42))
])


# Fitting the pipeline to the data
# excluding ticker column from the pipeline to avoid data leakage
clusters = pipeline.fit_predict(df_final_c.drop('Ticker', axis = 1))

df_final_c['Cluster'] = clusters

# Cluster analysis

In [46]:
clusters_df = df_final_c[['Ticker', 'Cluster']]

len(clusters_df)

30

In [47]:
print(len(clusters_df))
clusters_df.sort_values(by='Cluster')

30


Price,Ticker,Cluster
2,AMAT,0
5,ARM,0
7,CCEP,0
10,DDOG,0
12,HON,0
13,INSM,0
27,WDC,0
22,PCAR,0
15,KDP,1
11,FTNT,1


In [48]:
# Extract the exact row numbers of the 4 medoids
medoid_indices = pipeline.named_steps['kmedoids'].medoid_indices_

# find exact rows from DataFrame using .iloc
cluster_leaders = df_final.iloc[medoid_indices]

# fetch 'Ticker' column from specific rows and convert to list
representative_tickers = cluster_leaders['Ticker'].tolist()

print("--- Cluster Centers ---")
print(representative_tickers)

--- Cluster Centers ---
['ARM', 'MCHP', 'CEG', 'AAPL']


## Stocks chosen for prediction and forecasting due to being the closest to cluster center

1.   Paccar Inc. (PCAR)
2.   Applied Materials Inc. (AMAT)
3.   Microchip Technology Inc. (MCHP)
4.   Constellation Energy Corp (CEG)

